# 🔋 EV Battery Health Predictor & Failure Detection
### AI-Powered Battery Analytics with Smart Dashboard
---

## 📦 Cell 1 — Install Dependencies

In [ ]:
!pip install gradio scikit-learn pandas numpy plotly joblib -q

## 📥 Cell 2 — Imports & Dataset Loading

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
import joblib
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("✅ All libraries imported successfully!")

# ── Load Dataset ──────────────────────────────────────────────────────────────
print("\n📂 Loading dataset...")
df = pd.read_csv("ev_battery_complete-1.csv")
print(f"✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

✅ All libraries imported successfully!

📂 Loading dataset...
✅ Dataset loaded: 96,000 rows × 21 columns


,timestamp,SOC,voltage,current,temperature,charge_cycles,discharge_cycles,internal_resistance,capacity,SOH,...,discharge_rate,ambient_temp,voltage_variation,current_variation,temp_variation,efficiency,degradation_rate,over_temp_flag,over_voltage_flag,balancing_time
0,0,0.100000,3.7026,1.9791,24.021,1,0,0.05185,2.199,99.94,...,0.0,23.5,0.0084,0.0159,0.084,0.978,0.0,0,0,6.729128
1,1,0.114407,3.7168,1.9769,24.040,1,0,0.05181,2.199,99.94,...,0.0,23.5,0.0085,0.0159,0.084,0.978,0.0,0,0,6.809236
2,2,0.128814,3.7311,1.9749,24.057,1,0,0.05178,2.199,99.94,...,0.0,23.5,0.0085,0.0159,0.084,0.978,0.0,0,0,6.809236
3,3,0.143220,3.7453,1.9727,24.073,1,0,0.05175,2.199,99.94,...,0.0,23.5,0.0086,0.0159,0.085,0.978,0.0,0,0,6.889345
4,4,0.157627,3.7596,1.9707,24.087,1,0,0.05172,2.199,99.94,...,0.0,23.5,0.0086,0.0159,0.085,0.978,0.0,0,0,6.889345


## ⚙️ Cell 3 — Feature Engineering & Model Training

In [ ]:
# ── Failure Risk Label ────────────────────────────────────────────────────────
df['failure_risk'] = (
    (df['over_voltage_flag'] == 1) |
    (df['charge_cycles'] > 600) |
    (df['temperature'] > 35) |
    (df['SOH'] < 88)
).astype(int)

print(f"⚡ Failure risk cases : {df['failure_risk'].sum():,} "
      f"({df['failure_risk'].mean()*100:.2f}%)")

# ── Features ──────────────────────────────────────────────────────────────────
FEATURE_COLS = ['SOC', 'voltage', 'current', 'temperature',
                'charge_cycles', 'internal_resistance', 'capacity']

X             = df[FEATURE_COLS]
y_soh         = df['SOH']
y_failure     = df['failure_risk']

(X_train, X_test,
 y_soh_train, y_soh_test,
 y_fail_train, y_fail_test) = train_test_split(
    X, y_soh, y_failure,
    test_size=0.2, random_state=42, stratify=y_failure
)

# ── Train SOH Regressor ───────────────────────────────────────────────────────
print("\n🔧 Training SOH Predictor...")
soh_model = RandomForestRegressor(
    n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
soh_model.fit(X_train, y_soh_train)
soh_mae = mean_absolute_error(y_soh_test, soh_model.predict(X_test))
print(f"   ✅ SOH Model — MAE: {soh_mae:.4f}%")

# ── Train Failure Classifier ──────────────────────────────────────────────────
print("\n🔧 Training Failure Detection Model...")
fail_model = RandomForestClassifier(
    n_estimators=200, max_depth=12, random_state=42,
    n_jobs=-1, class_weight='balanced')
fail_model.fit(X_train, y_fail_train)
fail_acc = accuracy_score(y_fail_test, fail_model.predict(X_test))
print(f"   ✅ Failure Model — Accuracy: {fail_acc*100:.2f}%")

# ── Persist ───────────────────────────────────────────────────────────────────
joblib.dump(soh_model,  "soh_model.pkl")
joblib.dump(fail_model, "failure_model.pkl")
soh_model  = joblib.load("soh_model.pkl")
fail_model = joblib.load("failure_model.pkl")

print("\n🎉 Both models trained & saved successfully!")

⚡ Failure risk cases : 33,359 (34.75%)

🔧 Training SOH Predictor...
   ✅ SOH Model — MAE: 0.0000%

🔧 Training Failure Detection Model...
   ✅ Failure Model — Accuracy: 100.00%

🎉 Both models trained & saved successfully!


## 🧠 Cell 4 — Prediction & Dashboard Helper Functions

In [ ]:
# ── Prediction ────────────────────────────────────────────────────────────────
def predict_battery(SOC, voltage, current, temperature,
                    charge_cycles, internal_resistance, capacity):
    input_df = pd.DataFrame([{
        'SOC': SOC, 'voltage': voltage, 'current': current,
        'temperature': temperature, 'charge_cycles': charge_cycles,
        'internal_resistance': internal_resistance, 'capacity': capacity
    }])

    base_soh   = soh_model.predict(input_df)[0]
    degradation = min(8.0, charge_cycles * 0.008)
    predicted_soh = max(78.0, base_soh - degradation)

    fail_prob = fail_model.predict_proba(input_df)[0][1]

    # Health status
    if predicted_soh >= 95:
        health_status = "Excellent ✅"
    elif predicted_soh >= 90:
        health_status = "Good 🟢"
    elif predicted_soh >= 85:
        health_status = "Fair 🟠"
    else:
        health_status = "Poor — Replace Soon 🔴"

    # Risk level
    if fail_prob >= 0.65 or temperature > 38 or charge_cycles > 700:
        risk_level = "HIGH RISK ⚠️"
    elif fail_prob >= 0.35:
        risk_level = "MEDIUM RISK 🟠"
    else:
        risk_level = "LOW RISK ✅"

    temp_warning = (
        "🔴 OVERHEATING RISK!" if temperature > 35 else "✅ Normal Temperature"
    )

    # Gauge chart for SOH
    gauge_color = (
        "#27ae60" if predicted_soh >= 90
        else "#f39c12" if predicted_soh >= 85
        else "#e74c3c"
    )
    gauge = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=predicted_soh,
        delta={"reference": 100, "valueformat": ".1f"},
        title={"text": "State of Health (%)", "font": {"size": 18}},
        gauge={
            "axis": {"range": [75, 100], "tickwidth": 1},
            "bar": {"color": gauge_color, "thickness": 0.3},
            "steps": [
                {"range": [75, 85],  "color": "#fdecea"},
                {"range": [85, 90],  "color": "#fef3e2"},
                {"range": [90, 100], "color": "#eafaf1"},
            ],
            "threshold": {
                "line": {"color": "red", "width": 3},
                "thickness": 0.75,
                "value": 85
            }
        }
    ))
    gauge.update_layout(
        height=280,
        margin=dict(t=40, b=20, l=20, r=20),
        paper_bgcolor="rgba(0,0,0,0)"
    )

    # Failure probability bar
    bar_color = (
        "#e74c3c" if fail_prob >= 0.65
        else "#f39c12" if fail_prob >= 0.35
        else "#27ae60"
    )
    prob_fig = go.Figure(go.Bar(
        x=[fail_prob * 100],
        y=["Failure Probability"],
        orientation="h",
        marker_color=bar_color,
        text=[f"{fail_prob*100:.1f}%"],
        textposition="outside"
    ))
    prob_fig.update_layout(
        xaxis={"range": [0, 100], "title": "Probability (%)"},
        height=180,
        margin=dict(t=30, b=30, l=20, r=60),
        title="⚡ Failure Probability",
        paper_bgcolor="rgba(0,0,0,0)"
    )

    return (
        f"{predicted_soh:.2f}%",
        health_status,
        risk_level,
        temp_warning,
        f"{fail_prob*100:.1f}%",
        gauge,
        prob_fig
    )


# ── Dashboard ─────────────────────────────────────────────────────────────────
def create_dashboard():
    # 1. SOH degradation trend
    trend = df.groupby('charge_cycles')['SOH'].mean().reset_index()
    fig1 = px.line(
        trend, x='charge_cycles', y='SOH',
        title='📉 SOH Degradation Trend Over Charge Cycles',
        labels={'charge_cycles': 'Charge Cycles', 'SOH': 'State of Health (%)'},
        color_discrete_sequence=['#3498db']
    )
    fig1.update_layout(height=320, paper_bgcolor="rgba(0,0,0,0)")

    # 2. Temperature vs SOH scatter
    fig2 = px.scatter(
        df, x='temperature', y='SOH', color='failure_risk',
        title='🌡️ Temperature vs SOH  (1 = Failure Risk)',
        labels={'temperature': 'Temperature (°C)', 'SOH': 'SOH (%)'},
        color_continuous_scale=['#27ae60', '#f39c12', '#e74c3c'],
        opacity=0.6
    )
    fig2.update_layout(height=320, paper_bgcolor="rgba(0,0,0,0)")

    # 3. Temperature distribution
    fig3 = px.histogram(
        df, x='temperature', nbins=40,
        title='📊 Battery Temperature Distribution',
        labels={'temperature': 'Temperature (°C)', 'count': 'Count'},
        color_discrete_sequence=['#9b59b6']
    )
    fig3.update_layout(height=320, paper_bgcolor="rgba(0,0,0,0)")

    # 4. Failure risk pie
    risk_counts = df['failure_risk'].value_counts()
    fig4 = px.pie(
        names=['Healthy', 'At Risk'],
        values=[risk_counts.get(0, 0), risk_counts.get(1, 0)],
        title='🍩 Failure Risk Distribution',
        color_discrete_sequence=['#27ae60', '#e74c3c'],
        hole=0.4
    )
    fig4.update_layout(height=320, paper_bgcolor="rgba(0,0,0,0)")

    summary = f"""
## 📋 Dataset Summary
| Metric | Value |
|--------|-------|
| 📦 Total Records | **{len(df):,}** |
| 🔋 Average SOH | **{df['SOH'].mean():.2f}%** |
| ⚠️ Failure Risk Cases | **{df['failure_risk'].sum():,}** ({df['failure_risk'].mean()*100:.2f}%) |
| 🌡️ Max Temperature | **{df['temperature'].max():.1f}°C** |
| 🔄 Max Charge Cycles | **{int(df['charge_cycles'].max()):,}** |
| ⚡ Avg Voltage | **{df['voltage'].mean():.3f} V** |
"""
    return fig1, fig2, fig3, fig4, summary

print("✅ Helper functions defined.")

SyntaxError: invalid character '📦' (U+1F4E6) (4237675923.py, line 149)

## 🎨 Cell 5 — Launch Gradio UI

In [ ]:
# ── Custom CSS ────────────────────────────────────────────────────────────────
CUSTOM_CSS = """
/* ── Global ── */
body, .gradio-container {
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}

/* ── Header banner ── */
.header-banner {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    border-radius: 16px;
    padding: 28px 36px;
    margin-bottom: 20px;
    text-align: center;
    border: 1px solid rgba(255,255,255,0.08);
}
.header-banner h1 {
    color: #00d4ff !important;
    font-size: 2.2em;
    font-weight: 800;
    margin: 0 0 6px 0;
    letter-spacing: -0.5px;
}
.header-banner p {
    color: rgba(255,255,255,0.65) !important;
    font-size: 1em;
    margin: 0;
}

/* ── Metric cards ── */
.metric-card {
    background: linear-gradient(135deg, #1e293b, #0f172a) !important;
    border: 1px solid rgba(0,212,255,0.2) !important;
    border-radius: 12px !important;
    padding: 14px 18px !important;
}
.metric-card label {
    color: #94a3b8 !important;
    font-size: 0.78em !important;
    text-transform: uppercase;
    letter-spacing: 0.8px;
}
.metric-card textarea, .metric-card input {
    background: transparent !important;
    border: none !important;
    color: #e2e8f0 !important;
    font-size: 1.15em !important;
    font-weight: 700 !important;
    padding: 4px 0 !important;
}

/* ── Predict button ── */
.predict-btn button {
    background: linear-gradient(135deg, #00d4ff, #0099cc) !important;
    color: #0f172a !important;
    font-weight: 800 !important;
    font-size: 1.05em !important;
    border-radius: 10px !important;
    padding: 14px !important;
    border: none !important;
    transition: all 0.2s ease !important;
    box-shadow: 0 4px 20px rgba(0,212,255,0.3) !important;
}
.predict-btn button:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 30px rgba(0,212,255,0.45) !important;
}

/* ── Section headings ── */
.section-title {
    color: #00d4ff !important;
    font-size: 1em !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: 1.2px;
    border-bottom: 1px solid rgba(0,212,255,0.25);
    padding-bottom: 8px;
    margin-bottom: 14px !important;
}

/* ── Sliders & inputs ── */
.gr-slider input[type=range]::-webkit-slider-thumb {
    background: #00d4ff !important;
}

/* ── Tabs ── */
.tab-nav button {
    font-weight: 600 !important;
    font-size: 0.95em !important;
}
.tab-nav button.selected {
    border-bottom: 3px solid #00d4ff !important;
    color: #00d4ff !important;
}
"""

# ── Gradio App ────────────────────────────────────────────────────────────────
with gr.Blocks(
    title="🔋 EV Battery AI System",
    css=CUSTOM_CSS,
    theme=gr.themes.Base(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"]
    )
) as demo:

    # ── Header ──────────────────────────────────────────────────────────────
    gr.HTML("""
    <div class="header-banner">
        <h1>🔋 EV Battery Health AI System</h1>
        <p>Real-time State-of-Health Prediction &amp; Failure Risk Detection powered by Random Forest</p>
    </div>
    """)

    with gr.Tabs(elem_classes="main-tabs"):

        # ────────────────────────────────────────────────────────────────────
        #  TAB 1 — PREDICTOR
        # ────────────────────────────────────────────────────────────────────
        with gr.Tab("🔋 Battery Health Predictor"):

            gr.HTML('<p class="section-title">⚙️ Battery Parameters</p>')

            with gr.Row():
                # Left column — electrical params
                with gr.Column(scale=1):
                    gr.HTML('<p style="color:#94a3b8;font-size:0.82em;">⚡ Electrical Parameters</p>')
                    SOC = gr.Slider(
                        0, 100, value=70, step=1,
                        label="SOC — State of Charge (%)"
                    )
                    voltage = gr.Number(
                        value=3.85, precision=3,
                        label="Voltage (V)"
                    )
                    current = gr.Number(
                        value=1.5, precision=2,
                        label="Current (A)"
                    )
                    capacity = gr.Number(
                        value=2.10, precision=3,
                        label="Capacity (Ah)"
                    )

                # Right column — physical / operational params
                with gr.Column(scale=1):
                    gr.HTML('<p style="color:#94a3b8;font-size:0.82em;">🌡️ Physical & Operational Parameters</p>')
                    temperature = gr.Number(
                        value=32.0, precision=1,
                        label="Temperature (°C)"
                    )
                    charge_cycles = gr.Number(
                        value=450, precision=0,
                        label="Charge Cycles"
                    )
                    internal_resistance = gr.Number(
                        value=0.055, precision=4,
                        label="Internal Resistance (Ω)"
                    )

            # Predict button
            with gr.Row():
                predict_btn = gr.Button(
                    "🚀  Run AI Diagnosis",
                    variant="primary",
                    size="lg",
                    elem_classes="predict-btn"
                )

            gr.HTML('<p class="section-title" style="margin-top:20px;">📊 Prediction Results</p>')

            # Metric cards row
            with gr.Row():
                soh_out    = gr.Textbox(label="🔋 Predicted SOH",
                                        elem_classes="metric-card",
                                        interactive=False)
                health_out = gr.Textbox(label="💚 Health Status",
                                        elem_classes="metric-card",
                                        interactive=False)
                risk_out   = gr.Textbox(label="⚠️ Risk Level",
                                        elem_classes="metric-card",
                                        interactive=False)
                temp_out   = gr.Textbox(label="🌡️ Temperature",
                                        elem_classes="metric-card",
                                        interactive=False)
                prob_out   = gr.Textbox(label="📉 Failure Probability",
                                        elem_classes="metric-card",
                                        interactive=False)

            # Chart row
            with gr.Row():
                gauge_out = gr.Plot(label="SOH Gauge")
                prob_chart = gr.Plot(label="Failure Probability Chart")

            predict_btn.click(
                fn=predict_battery,
                inputs=[SOC, voltage, current, temperature,
                        charge_cycles, internal_resistance, capacity],
                outputs=[soh_out, health_out, risk_out, temp_out,
                         prob_out, gauge_out, prob_chart]
            )

        # ────────────────────────────────────────────────────────────────────
        #  TAB 2 — SMART DASHBOARD
        # ────────────────────────────────────────────────────────────────────
        with gr.Tab("📊 Smart Dashboard"):

            with gr.Row():
                gr.HTML('<p class="section-title">📡 Live Analytics & Visualizations</p>')
                refresh_btn = gr.Button(
                    "🔄 Refresh",
                    variant="secondary",
                    size="sm"
                )

            with gr.Row():
                plot1 = gr.Plot()
                plot2 = gr.Plot()
            with gr.Row():
                plot3 = gr.Plot()
                plot4 = gr.Plot()

            summary_md = gr.Markdown()

            refresh_btn.click(
                fn=create_dashboard,
                inputs=None,
                outputs=[plot1, plot2, plot3, plot4, summary_md]
            )
            demo.load(
                fn=create_dashboard,
                inputs=None,
                outputs=[plot1, plot2, plot3, plot4, summary_md]
            )

        # ────────────────────────────────────────────────────────────────────
        #  TAB 3 — ABOUT
        # ────────────────────────────────────────────────────────────────────
        with gr.Tab("ℹ️ About"):
            gr.Markdown("""
## About This App

### 🔋 EV Battery Health Predictor & Failure Detection

This application uses **Machine Learning (Random Forest)** to predict the health and failure risk of
Electric Vehicle (EV) batteries in real time.

---

### 🧠 Models Used
| Model | Type | Purpose |
|-------|------|---------|
| **SOH Predictor** | Random Forest Regressor | Predicts State of Health (%) |
| **Failure Detector** | Random Forest Classifier | Estimates failure probability |

### 📥 Input Features
| Feature | Description |
|---------|-------------|
| **SOC** | State of Charge (%) |
| **Voltage** | Battery terminal voltage (V) |
| **Current** | Charge / discharge current (A) |
| **Temperature** | Cell temperature (°C) |
| **Charge Cycles** | Total completed charge cycles |
| **Internal Resistance** | Cell internal resistance (Ω) |
| **Capacity** | Actual cell capacity (Ah) |

### ⚠️ Failure Risk Criteria
A battery is flagged **at risk** when ANY of the following is true:
- Over-voltage flag is active
- Charge cycles > 600
- Temperature > 35 °C
- State of Health < 88%

---
*Built with ❤️ using Scikit-Learn, Gradio & Plotly*
""")

    gr.HTML("""
    <div style="text-align:center;padding:18px 0 8px;color:#475569;font-size:0.82em;">
        🔋 EV Battery AI System &nbsp;|&nbsp;
        Random Forest Models &nbsp;|&nbsp;
        Built with Gradio &amp; Plotly
    </div>
    """)

demo.launch(
    share=False,       # set True to get a public URL
    inbrowser=False,   # set True to auto-open in browser
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>